<a href="https://colab.research.google.com/github/sabthe02/Obligatorio-IA/blob/Sabine/obligatorio_hotel_cancelaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicción de cancelaciones hoteleras
**Electiva Inteligencia Artificial | Licenciatura en Informática | UDE**

**Integrantes:** Sabine Theiss, Jorge Avellanal y Mateo Mendiondo.

**Datos:** Hotel Booking Demand. El informe Word contiene la explicación completa, las decisiones metodológicas y los resultados.

## 0. Preparación del entorno

In [ ]:
import random


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

from IPython.display import display as mostrar

ETIQUETAS_COLUMNAS = {
    "analysis": "análisis", "model": "modelo",
    "average_precision_mean": "AP media", "average_precision_std": "desviación estándar de AP",
    "random_baseline_average_precision": "AP del baseline aleatorio",
    "fraction_alerted": "fracción alertada", "precision_at_k": "precisión@k",
    "recall_at_k": "exhaustividad@k", "alerts_per_1000": "alertas por cada 1.000 reservas",
    "random_precision_reference": "precisión aleatoria de referencia",
    "random_recall_reference": "exhaustividad aleatoria de referencia",
    "outer_fold": "partición externa", "candidate": "candidato", "parameters": "parámetros",
    "inner_average_precision_mean": "AP media interna",
    "inner_average_precision_std": "desviación estándar interna de AP", "selected": "seleccionado",
    "fold": "partición", "average_precision": "AP", "n_features": "n.º de variables",
    "selected_transformed_features": "variables transformadas seleccionadas",
    "delta_vs_full": "diferencia frente al modelo completo",
    "delta_vs_honest": "diferencia frente al modelo sin fuga",
    "brier_score": "Brier", "log_loss": "pérdida logarítmica",
    "validation_rows": "reservas de validación", "scenario": "escenario",
    "cost_fp": "costo de falsos positivos", "cost_fn": "costo de falsos negativos",
    "threshold": "umbral", "false_positives": "falsos positivos",
    "false_negatives": "falsos negativos", "normalized_cost_per_reservation": "costo relativo por reserva",
    "metric": "métrica", "estimate": "estimación", "ci_2_5%": "IC 2,5 %",
    "ci_97_5%": "IC 97,5 %", "valid_replicates": "réplicas válidas",
    "roc_auc": "ROC-AUC", "precision": "precisión", "recall": "exhaustividad",
    "f1": "F1", "test_rows": "reservas de prueba", "group": "grupo", "value": "valor",
    "rows": "reservas", "precision_at_10_percent": "precisión en el 10 %",
    "recall_at_10_percent": "exhaustividad en el 10 %", "feature": "variable",
    "importance_mean": "importancia media",
    "importance_std_between_folds": "desviación estándar entre particiones",
    "cancellation_probability": "probabilidad de cancelación",
    "predicted_cancellation": "cancelación predicha", "y_true": "cancelación real",
    "prop_nulos": "proporción de valores ausentes", "proporcion": "proporción",
    "no_cancelada": "no cancelada", "cancelada": "cancelada",
    "accuracy": "exactitud", "macro avg": "promedio macro", "weighted avg": "promedio ponderado",
}

ETIQUETAS_VALORES = {
    "main": "principal", "deduplicated": "sin duplicados",
    "Dummy baseline": "baseline aleatorio", "Logistic regression": "regresión logística",
    "Random forest": "Random Forest", "Gradient boosting": "Gradient Boosting",
    "RBF Nyström + linear SVM (approximation)": "SVM lineal aproximada con kernel RBF mediante Nyström",
    "average_precision": "AP", "roc_auc": "ROC-AUC", "precision": "precisión",
    "recall": "exhaustividad", "f1": "F1", "brier_score": "Brier",
    "log_loss": "pérdida logarítmica", "accuracy": "exactitud",
    "macro avg": "promedio macro", "weighted avg": "promedio ponderado",
}

def mostrar_en_espanol(tabla):
    if isinstance(tabla, pd.DataFrame):
        tabla = tabla.rename(columns=ETIQUETAS_COLUMNAS, index=ETIQUETAS_VALORES)
        for columna in tabla.select_dtypes(include=["object", "string"]).columns:
            tabla[columna] = tabla[columna].replace(ETIQUETAS_VALORES)
    elif isinstance(tabla, pd.Series):
        tabla = tabla.rename(ETIQUETAS_COLUMNAS.get(tabla.name, tabla.name))
        tabla = tabla.replace(ETIQUETAS_VALORES)
    mostrar(tabla)

Genera el archivo de dependencias del entorno actual.

In [ ]:
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

if importlib.util.find_spec("docx") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])

REQUIRED_DISTRIBUTIONS = [
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "joblib",
    "python-docx",
]
missing_distributions = []
requirements_lines = []
for distribution in REQUIRED_DISTRIBUTIONS:
    try:
        requirements_lines.append(f"{distribution}=={version(distribution)}")
    except PackageNotFoundError:
        missing_distributions.append(distribution)

if missing_distributions:
    raise RuntimeError(
        "Instala estas dependencias antes de generar requirements.txt y el informe DOCX: "
        + ", ".join(missing_distributions)
        + ". En Colab puedes ejecutar %pip install "
        + " ".join(missing_distributions)
        + " y volver a correr esta celda."
    )

Path("requirements.txt").write_text(
    "\n".join(requirements_lines) + "\n",
    encoding="utf-8",
)
print("Archivo requirements.txt creado con las versiones instaladas:")
print("\n".join(requirements_lines))

## 1. Objetivo
Predice `is_canceled` con información disponible al crear la reserva; el informe explica el uso previsto y sus límites.

## 2. Datos y partición temporal
Define la partición antes del análisis y mantiene separado el conjunto de prueba.

In [ ]:
LOCAL_DATA_PATH = Path("hotel_bookings.csv")
URL = "https://raw.githubusercontent.com/sabthe02/Obligatorio-IA/main/hotel_bookings.csv"
if LOCAL_DATA_PATH.exists():
    df_raw = pd.read_csv(LOCAL_DATA_PATH)
    print(f"Dataset cargado desde el repositorio local: {LOCAL_DATA_PATH}")
else:
    df_raw = pd.read_csv(URL)
    print(f"Dataset cargado desde GitHub: {URL}")
print("Filas y columnas del conjunto de datos:", df_raw.shape)
df_raw.head()

In [ ]:
MESES = {m: i for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June",
     "July", "August", "September", "October", "November", "December"], start=1)}

df = df_raw.copy()
df["arrival_date"] = pd.to_datetime(dict(
    year=df["arrival_date_year"],
    month=df["arrival_date_month"].map(MESES),
    day=df["arrival_date_day_of_month"],
))
assert df["arrival_date"].notna().all(), "Hay fechas de llegada que no se pudieron construir"

print("Rango de fechas de llegada:", df["arrival_date"].min().date(), "->", df["arrival_date"].max().date())

In [ ]:
# El tramo temporal más reciente se reserva para evaluar la generalización hacia el futuro.
TARGET = "is_canceled"
TEST_FRAC = 0.20

# La fecha de reserva no está disponible; se aproxima restando lead_time a la llegada.
df["booking_date_est"] = df["arrival_date"] - pd.to_timedelta(df["lead_time"], unit="D")
df = df.sort_values("booking_date_est", kind="mergesort").reset_index(drop=True)
cut_idx = int(len(df) * (1 - TEST_FRAC))
booking_cutoff = df.loc[cut_idx, "booking_date_est"]
# Solo se entrena con reservas anteriores al corte cuyo resultado ya habría madurado.
train_mask = (df["booking_date_est"] < booking_cutoff) & (df["arrival_date"] < booking_cutoff)
# El holdout agrupa las reservas cuya fecha estimada cae desde el corte en adelante.
test_mask = df["booking_date_est"] >= booking_cutoff
# Se excluyen del train las reservas anteriores al corte que todavía no habrían madurado.
purged_mask = (df["booking_date_est"] < booking_cutoff) & (df["arrival_date"] >= booking_cutoff)

train_df = df[train_mask].copy()
test_df = df[test_mask].copy()
purged_count = int(purged_mask.sum())

print(f"Corte por fecha estimada de reserva: {booking_cutoff.date()}")
print(f"Entrenamiento maduro: {len(train_df):,} reservas; fechas de reserva {train_df['booking_date_est'].min().date()} -> {train_df['booking_date_est'].max().date()}")
print(f"Prueba futura: {len(test_df):,} reservas; fechas de reserva {test_df['booking_date_est'].min().date()} -> {test_df['booking_date_est'].max().date()}")
print(f"Reservas excluidas por llegada o resultado posterior al corte: {purged_count:,} reservas")
print("Proporción de cancelaciones en entrenamiento:", f"{train_df[TARGET].mean():.1%}")

# booking_date_est solo se usa para separar el tiempo, no como predictor.
test_df.to_csv("test_holdout.csv", index=False)
del test_df, df, df_raw, train_mask, test_mask, purged_mask

Aplica un corte temporal aproximado y excluye reservas cuyo resultado aún no habría madurado.

## 3. Exploración de datos
Resume la calidad y las relaciones descriptivas usando solo entrenamiento.

In [ ]:
print("Distribución de la variable objetivo en entrenamiento:")
mostrar_en_espanol(train_df[TARGET].value_counts(normalize=True).rename_axis("resultado").rename("proporción"))

nulls = train_df.isna().mean().sort_values(ascending=False)
print("\nVariables con valores ausentes:")
mostrar_en_espanol(nulls[nulls > 0].to_frame("prop_nulos"))

# Se comparan las proporciones de faltantes por clase usando únicamente el conjunto de entrenamiento.
missingness_by_target = pd.DataFrame({
    "no_cancelada": train_df.loc[train_df[TARGET] == 0].isna().mean(),
    "cancelada": train_df.loc[train_df[TARGET] == 1].isna().mean(),
})
missingness_by_target = missingness_by_target.loc[missingness_by_target.sum(axis=1) > 0]
print("Proporción de valores ausentes según el resultado:")
mostrar_en_espanol(missingness_by_target)

print("Duplicados exactos en entrenamiento:", train_df.duplicated().sum())

Muestra la proporción de reservas canceladas y no canceladas.

In [ ]:
target_share = train_df[TARGET].value_counts(normalize=True).sort_index()
ax = target_share.plot.bar(color=["#4C78A8", "#F58518"], rot=0)
ax.set_xticklabels(["No cancelada", "Cancelada"])
ax.set_ylabel("Proporción")
ax.set_title("Distribución del target en train")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
plt.tight_layout()
plt.show()

Visualiza el balance de clases del conjunto de entrenamiento.

Compara la tasa de cancelación entre tipos de hotel.

In [ ]:
hotel_rates = train_df.groupby("hotel", observed=True)[TARGET].mean().sort_values()
ax = hotel_rates.plot.bar(color="#4C78A8", rot=0)
ax.set_ylabel("Tasa de cancelación")
ax.set_title("Cancelación por tipo de hotel (train)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
plt.tight_layout()
plt.show()

Compara las tasas mensuales de cancelación.

Explora la tasa de cancelación según el mes de llegada.

In [ ]:
month_rates = train_df.groupby(train_df["arrival_date"].dt.month)[TARGET].mean()
ax = month_rates.plot(marker="o", color="#54A24B")
ax.set_xticks(range(1, 13))
ax.set_xlabel("Mes de llegada")
ax.set_ylabel("Tasa de cancelación")
ax.set_title("Cancelación por mes de llegada (train)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
plt.tight_layout()
plt.show()

Compara la cancelación por anticipación de la reserva.

Agrupa las reservas por anticipación para explorar posibles diferencias.

In [ ]:
lead_bins = [-1, 7, 30, 90, 180, 365, np.inf]
lead_labels = ["0-7", "8-30", "31-90", "91-180", "181-365", "366+"]
lead_band = pd.cut(train_df["lead_time"], bins=lead_bins, labels=lead_labels)
lead_rates = train_df.groupby(lead_band, observed=True)[TARGET].mean()
ax = lead_rates.plot.bar(color="#E45756", rot=30)
ax.set_xlabel("Días entre reserva y llegada")
ax.set_ylabel("Tasa de cancelación")
ax.set_title("Cancelación según anticipación (train)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
plt.tight_layout()
plt.show()

Muestra la cancelación según el tipo de depósito.

Compara descriptivamente las condiciones de depósito.

In [ ]:
deposit_rates = train_df.groupby("deposit_type", observed=True)[TARGET].agg(["mean", "size"]).sort_values("mean")
ax = deposit_rates["mean"].plot.bar(color="#B279A2", rot=20)
ax.set_ylabel("Tasa de cancelación")
ax.set_title("Cancelación por tipo de depósito (train)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
for position, (rate, count) in enumerate(zip(deposit_rates["mean"], deposit_rates["size"])):
    ax.annotate(f"n={count:,}", (position, rate), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

Resume la distribución del precio diario de la habitación por resultado.

Compara las distribuciones de ADR, limitando la escala para facilitar su lectura.

In [ ]:
adr_plot = train_df[[TARGET, "adr"]].dropna().copy()
adr_upper = adr_plot["adr"].quantile(0.99)
adr_plot.boxplot(column="adr", by=TARGET, showfliers=False, grid=False)
plt.ylim(0, adr_upper)
plt.xlabel("Cancelada (0 = no, 1 = sí)")
plt.ylabel("ADR (recortado visualmente al percentil 99)")
plt.title("ADR según resultado de la reserva")
plt.suptitle("")
plt.tight_layout()
plt.show()

Explora la distribución de ADR para reservas canceladas y no canceladas.

## 4. Fuga de datos y validación
Audita la disponibilidad de las variables y define validación temporal.

In [ ]:
# La auditoría documenta disponibilidad temporal y riesgo de fuga para cada variable original.
audit_rows = [
    # variable, disponible al reservar, riesgo de proxy, decisión, justificación
    ("hotel", "Sí", "Bajo", "Usar", "Tipo de hotel (resort o ciudad), conocido al reservar."),
    ("is_canceled", "-", "-", "Target", "Variable a predecir."),
    ("lead_time", "Sí", "Bajo", "Usar", "Días entre la reserva y la llegada; se conoce al reservar."),
    ("arrival_date_year", "Sí", "Bajo", "Transformar o descartar", "Con partición temporal el test cae en años/fechas no vistas; puede no generalizar."),
    ("arrival_date_month", "Sí", "Bajo", "Transformar", "Captura estacionalidad; codificar como número o de forma cíclica."),
    ("arrival_date_week_number", "Sí", "Bajo", "Evaluar", "Redundante con el mes; decidir en la selección de variables."),
    ("arrival_date_day_of_month", "Sí", "Bajo", "Evaluar", "Poco aporte esperado; medir en la ablation."),
    ("stays_in_weekend_nights", "Sí", "Bajo", "Usar", "Estadía planificada; verificar que no refleje cambios posteriores."),
    ("stays_in_week_nights", "Sí", "Bajo", "Usar", "Estadía planificada; verificar que no refleje cambios posteriores."),
    ("adults", "Sí", "Bajo", "Usar", "Composición de la reserva."),
    ("children", "Sí", "Bajo", "Usar", "Composición de la reserva; tiene pocos faltantes."),
    ("babies", "Sí", "Bajo", "Usar", "Composición de la reserva."),
    ("meal", "Sí", "Bajo", "Usar", "Régimen de comidas contratado."),
    ("country", "Sí", "Bajo", "Transformar", "Alta cardinalidad y faltantes: frecuencia o agrupación de categorías raras."),
    ("market_segment", "Sí", "Bajo", "Usar", "Segmento de mercado; revisar categorías raras o 'Undefined'."),
    ("distribution_channel", "Sí", "Bajo", "Usar", "Canal de distribución; revisar categorías raras o 'Undefined'."),
    ("is_repeated_guest", "Sí", "Bajo", "Usar", "Dato del huésped previo a la reserva."),
    ("previous_cancellations", "Sí", "Bajo", "Usar", "Historial del cliente anterior a esta reserva."),
    ("previous_bookings_not_canceled", "Sí", "Bajo", "Usar", "Historial del cliente anterior a esta reserva."),
    ("reserved_room_type", "Sí", "Bajo", "Usar", "Tipo de habitación reservada."),
    ("assigned_room_type", "Dudosa", "Alto", "Descartar", "Se asigna cerca de la estadía y puede diferir de la reservada; candidata a fuga y se excluye del modelo principal."),
    ("booking_changes", "No", "Alto", "Descartar", "Cuenta cambios posteriores a la creación de la reserva; no está disponible al crear la reserva."),
    ("deposit_type", "Dudosa", "Medio", "Excluir del modelo principal", "La disponibilidad exacta al reservar no está confirmada; se excluye porque su disponibilidad al reservar no está confirmada."),
    ("agent", "Sí", "Bajo", "Transformar", "Alta cardinalidad y faltantes: frecuencia, agrupación e indicador de faltante."),
    ("company", "Sí", "Bajo", "Transformar", "Muchos faltantes: indicador de presencia, agrupación."),
    ("days_in_waiting_list", "Dudosa", "Medio", "Excluir del modelo principal", "La disponibilidad al reservar no está confirmada; se excluye porque su disponibilidad al reservar no está confirmada."),
    ("customer_type", "Sí", "Bajo", "Usar", "Tipo de cliente (contrato, grupo, transient)."),
    ("adr", "Dudosa", "Medio", "Excluir del modelo principal", "La disponibilidad y cálculo al reservar no están confirmados; se excluye porque su disponibilidad al reservar no está confirmada."),
    ("required_car_parking_spaces", "Dudosa", "Medio", "Excluir del modelo principal", "La disponibilidad al reservar no está confirmada; se excluye porque su disponibilidad al reservar no está confirmada."),
    ("total_of_special_requests", "Dudosa", "Medio", "Excluir del modelo principal", "La disponibilidad al reservar no está confirmada; se excluye porque su disponibilidad al reservar no está confirmada."),
    ("reservation_status", "No", "Muy alto", "Descartar", "Fuga directa: informa si la reserva fue cancelada, hizo check-out o no se presentó. Solo para el experimento de fuga."),
    ("reservation_status_date", "No", "Muy alto", "Descartar", "Fecha del último estado de la reserva; fuga. Solo para el experimento de fuga."),
]

audit = pd.DataFrame(audit_rows, columns=[
    "variable", "disponible_al_reservar", "riesgo_proxy", "decision", "justificacion"])

# Control de cobertura: toda columna del dataset debe estar auditada.
cols_dataset = set(train_df.columns) - {"arrival_date", "booking_date_est"}  # Fechas derivadas
sin_auditar = cols_dataset - set(audit["variable"])
sobran = set(audit["variable"]) - cols_dataset
print("Sin auditar:", sin_auditar or "ninguna")
print("En la tabla pero no en el dataset:", sobran or "ninguna")

audit

In [ ]:
# Columnas que se usan SOLO en el experimento de fuga (sección 6.4), nunca en el modelo real.
LEAK_COLS = ["reservation_status", "reservation_status_date"]

Preprocesa las variables dentro del pipeline y de cada partición de entrenamiento.

## 5. Variables derivadas
Construye predictores disponibles al reservar; el informe documenta su definición y motivación.

In [ ]:
# Estas transformaciones dependen solo de la reserva y no aprenden parámetros de la muestra.
def add_stateless_features(d):
    """Deriva variables de estadía, composición, calendario e historial previo."""
    d = d.copy()
    d["total_nights"] = d["stays_in_weekend_nights"] + d["stays_in_week_nights"]
    d["total_guests"] = d["adults"] + d["children"].fillna(0) + d["babies"]
    d["has_children"] = ((d["children"].fillna(0) + d["babies"]) > 0).astype(int)
    d["arrival_month"] = d["arrival_date"].dt.month
    d["arrival_dow"] = d["arrival_date"].dt.day_of_week
    d["arrival_is_weekend"] = d["arrival_dow"].isin([5, 6]).astype(int)
    d["arrival_month_sin"] = np.sin(2 * np.pi * d["arrival_month"] / 12)
    d["arrival_month_cos"] = np.cos(2 * np.pi * d["arrival_month"] / 12)
    d["lead_time_band"] = pd.cut(
        d["lead_time"],
        bins=[-1, 7, 30, 90, 180, 365, np.inf],
        labels=["0-7", "8-30", "31-90", "91-180", "181-365", "366+"],
    ).astype("object")
    prior_bookings = d["previous_cancellations"] + d["previous_bookings_not_canceled"]
    d["prev_cancel_ratio"] = d["previous_cancellations"].div(prior_bookings.where(prior_bookings > 0)).fillna(0)
    d["adr_per_guest"] = d["adr"].div(d["total_guests"].where(d["total_guests"] > 0))
    return d

train_df = add_stateless_features(train_df)

print("Variables derivadas:", ["noches totales", "cantidad de huéspedes", "presencia de niños", "mes de llegada", "día de la semana de llegada", "llegada en fin de semana", "mes de llegada (seno)", "mes de llegada (coseno)", "tramo de anticipación", "proporción de cancelaciones previas", "precio diario por huésped"])
print("Dimensiones del entrenamiento con variables derivadas:", train_df.shape)

### Sensibilidad a duplicados
Compara el entrenamiento completo con una variante sin duplicados exactos.

In [ ]:
# Las variables sin disponibilidad confirmada y las posteriores a la reserva se excluyen del modelo válido.
BOOKING_TIME_UNVERIFIED_COLS = ["deposit_type", "adr", "days_in_waiting_list", "required_car_parking_spaces", "total_of_special_requests"]
POST_BOOKING_COLS = ["booking_changes", "assigned_room_type"]
MODEL_EXCLUDE_COLS = ["arrival_date", "booking_date_est", "reservation_status", "reservation_status_date", *POST_BOOKING_COLS, *BOOKING_TIME_UNVERIFIED_COLS, "adr_per_guest"]
MODEL_FEATURE_COLS = [col for col in train_df.columns if col not in MODEL_EXCLUDE_COLS + [TARGET]]
assert not set(MODEL_FEATURE_COLS).intersection(MODEL_EXCLUDE_COLS + [TARGET])

train_df_dedup_sensitivity = train_df.drop_duplicates().copy()
duplicates_removed = len(train_df) - len(train_df_dedup_sensitivity)
assert not train_df_dedup_sensitivity.duplicated().any()

print(f"Entrenamiento principal: {len(train_df):,} filas; tasa de cancelación {train_df[TARGET].mean():.2%}")
print(f"Variante sin duplicados: {len(train_df_dedup_sensitivity):,} filas; tasa de cancelación {train_df_dedup_sensitivity[TARGET].mean():.2%}")
print(f"Filas duplicadas retiradas para el análisis de sensibilidad: {duplicates_removed:,}")

## 6. Modelado
Compara modelos mediante validación temporal; el informe presenta criterios y resultados.

In [ ]:
def capacity_scenario_metrics(y_true, y_score, fractions=(0.05, 0.10, 0.20)):
    """Resume el ranking como si se alertara al porcentaje de reservas de mayor riesgo."""
    from sklearn.metrics import average_precision_score

    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score, dtype=float)
    if y_true.ndim != 1 or y_score.ndim != 1 or len(y_true) != len(y_score) or len(y_true) == 0:
        raise ValueError("y_true and y_score must be non-empty one-dimensional arrays of equal length")
    if not np.isin(y_true, [0, 1]).all() or not np.isfinite(y_score).all():
        raise ValueError("y_true must contain only 0/1 labels and y_score must be finite")
    positive_count = int(y_true.sum())
    if positive_count == 0:
        raise ValueError("At least one positive label is required to calculate recall")

    fractions = tuple(float(fraction) for fraction in fractions)
    if not fractions or any(not np.isfinite(fraction) or fraction <= 0 or fraction > 1 for fraction in fractions):
        raise ValueError("fractions must contain values in the interval (0, 1]")

    order = np.argsort(-y_score, kind="mergesort")
    rows = []
    for fraction in fractions:
        k = min(len(y_true), max(1, int(np.ceil(fraction * len(y_true)))))
        selected_positives = int(y_true[order[:k]].sum())
        rows.append({
            "fraction_alerted": fraction,
            "alerts_per_1000": k / len(y_true) * 1000,
            "precision_at_k": selected_positives / k,
            "recall_at_k": selected_positives / positive_count,
            "random_precision_reference": positive_count / len(y_true),
            "random_recall_reference": k / len(y_true),
        })
    return average_precision_score(y_true, y_score), pd.DataFrame(rows)

def make_purged_rolling_folds(data, n_splits=5, initial_train_fraction=0.5, validation_windows=None):
    """Crea folds cronológicos y purga del train reservas aún inmaduras al inicio de cada ventana."""
    if n_splits < 1 or not 0 < initial_train_fraction < 1:
        raise ValueError("n_splits must be positive and initial_train_fraction must be between 0 and 1")
    if data[["booking_date_est", "arrival_date", TARGET]].isna().any().any():
        raise ValueError("booking_date_est, arrival_date, and target must not contain missing values")

    if validation_windows is None:
        unique_dates, date_counts = np.unique(data["booking_date_est"].to_numpy(), return_counts=True)
        if len(unique_dates) <= n_splits:
            raise ValueError("Not enough distinct booking dates for the requested folds")
        first_validation = int(np.searchsorted(np.cumsum(date_counts), np.ceil(initial_train_fraction * len(data)), side="left") + 1)
        first_validation = min(max(first_validation, 1), len(unique_dates) - n_splits)
        remaining_dates = unique_dates[first_validation:]
        remaining_counts = date_counts[first_validation:]
        cumulative_counts = np.cumsum(remaining_counts)
        targets = np.arange(1, n_splits) * cumulative_counts[-1] / n_splits
        split_positions = np.searchsorted(cumulative_counts, targets, side="left") + 1
        if len(np.unique(split_positions)) != n_splits - 1:
            raise ValueError("Not enough distinct dates to create non-empty, approximately balanced validation folds")
        date_blocks = np.split(remaining_dates, split_positions)
        validation_windows = [(pd.Timestamp(block[0]), pd.Timestamp(block[-1])) for block in date_blocks]
    else:
        validation_windows = [(pd.Timestamp(start), pd.Timestamp(end)) for start, end in validation_windows]
        if len(validation_windows) != n_splits:
            raise ValueError("validation_windows must contain exactly n_splits date ranges")

    folds = []
    summary = []
    booking_dates = data["booking_date_est"]
    arrival_dates = data["arrival_date"]
    for fold_number, (start, end) in enumerate(validation_windows, start=1):
        if start > end:
            raise ValueError("Each validation window must start on or before it ends")
        train_mask = booking_dates.lt(start) & arrival_dates.lt(start)
        validation_mask = booking_dates.between(start, end, inclusive="both")
        purged_mask = booking_dates.lt(start) & arrival_dates.ge(start)
        train_indices = np.flatnonzero(train_mask.to_numpy())
        validation_indices = np.flatnonzero(validation_mask.to_numpy())
        if not len(train_indices) or not len(validation_indices):
            raise ValueError(f"Fold {fold_number} has an empty training or validation partition")
        if data.iloc[validation_indices][TARGET].sum() == 0:
            raise ValueError(f"Fold {fold_number} has no positive labels for Average Precisión/recall")

        train_hashes = pd.util.hash_pandas_object(data.iloc[train_indices], index=False).to_numpy()
        validation_hashes = pd.util.hash_pandas_object(data.iloc[validation_indices], index=False).to_numpy()
        if np.intersect1d(train_hashes, validation_hashes).size:
            raise AssertionError("Exact duplicate rows must not cross a fold boundary")

        folds.append((train_indices, validation_indices))
        summary.append({
            "fold": fold_number,
            "validation_start": start,
            "validation_end": end,
            "train_rows": len(train_indices),
            "validation_rows": len(validation_indices),
            "purged_early_bookings": int(purged_mask.sum()),
        })
    return folds, pd.DataFrame(summary), validation_windows

# Se generan las ventanas una vez con el train principal y se reutilizan para el análisis de sensibilidad deduplicado.
temporal_folds, temporal_fold_summary, temporal_validation_windows = make_purged_rolling_folds(train_df)
temporal_fold_summary

def build_model_suite(numeric_features, categorical_features):
    """Construye preprocesamiento y modelos para una comparación con la misma partición temporal."""
    from sklearn.compose import ColumnTransformer
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.kernel_approximation import Nystroem
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.svm import LinearSVC

    # La imputación, el escalado y la codificación se ajustan dentro de cada fold del pipeline.
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([
                ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                ("scaler", StandardScaler(with_mean=False)),
            ]), numeric_features),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent", add_indicator=True)),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), categorical_features),
        ],
        sparse_threshold=0.3,
    )

    def make_pipeline(classifier, approximate_rbf=False):
        steps = [("preprocess", preprocessor)]
        if approximate_rbf:
            steps.append(("rbf_approximation", Nystroem(kernel="rbf", n_components=200, random_state=SEED)))
        steps.append(("classifier", classifier))
        return Pipeline(steps)

    return {
        "Dummy baseline": DummyClassifier(strategy="prior"),
        "Logistic regression": make_pipeline(LogisticRegression(
            max_iter=3000, solver="liblinear", tol=1e-3,
            class_weight="balanced", random_state=SEED)),
        "Random forest": make_pipeline(RandomForestClassifier(
            n_estimators=100, max_features="sqrt", min_samples_leaf=2,
            class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)),
        "Gradient boosting": make_pipeline(GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.05, max_depth=2,
            min_samples_leaf=20, subsample=0.8, random_state=SEED)),
        "RBF Nyström + linear SVM (approximation)": make_pipeline(
            LinearSVC(class_weight="balanced", dual="auto", max_iter=3000, random_state=SEED),
            approximate_rbf=True),
    }

def evaluate_temporal_model_suite(data, folds, models, analysis_name, validation_data=None, validation_folds=None):
    """Ajusta copias independientes por fold y calcula métricas solo en sus filas de validación."""
    from sklearn.base import clone
    from sklearn.dummy import DummyClassifier
    from sklearn.metrics import average_precision_score

    validation_data = data if validation_data is None else validation_data
    validation_folds = folds if validation_folds is None else validation_folds
    if len(folds) != len(validation_folds):
        raise ValueError("Training and validation fold lists must have equal length")
    X_train_data = data[MODEL_FEATURE_COLS]
    y_train_data = data[TARGET].to_numpy(dtype=int)
    X_validation_data = validation_data[MODEL_FEATURE_COLS]
    y_validation_data = validation_data[TARGET].to_numpy(dtype=int)
    fold_rows = []
    capacity_rows = []
    for model_name, model in models.items():
        for fold_number, ((train_indices, _), (_, validation_indices)) in enumerate(zip(folds, validation_folds), start=1):
            X_train = X_train_data.iloc[train_indices]
            X_validation = X_validation_data.iloc[validation_indices]
            y_train = y_train_data[train_indices]
            y_validation = y_validation_data[validation_indices]
            fitted_model = clone(model)
            if isinstance(fitted_model, DummyClassifier):
                fitted_model.fit(np.zeros((len(train_indices), 1)), y_train)
                scores = fitted_model.predict_proba(np.zeros((len(validation_indices), 1)))[:, 1]
            else:
                fitted_model.fit(X_train, y_train)
                if hasattr(fitted_model, "predict_proba"):
                    scores = fitted_model.predict_proba(X_validation)[:, 1]
                else:
                    scores = fitted_model.decision_function(X_validation)

            fold_rows.append({
                "analysis": analysis_name,
                "model": model_name,
                "fold": fold_number,
                "average_precision": average_precision_score(y_validation, scores),
                "random_baseline_average_precision": y_validation.mean(),
                "train_rows": len(train_indices),
                "validation_rows": len(validation_indices),
            })
            if not isinstance(fitted_model, DummyClassifier):
                _, scenarios = capacity_scenario_metrics(y_validation, scores)
                scenarios["analysis"] = analysis_name
                scenarios["model"] = model_name
                scenarios["fold"] = fold_number
                capacity_rows.append(scenarios)

    return pd.DataFrame(fold_rows), pd.concat(capacity_rows, ignore_index=True)

model_input = train_df[MODEL_FEATURE_COLS]
categorical_features = model_input.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_features = model_input.select_dtypes(include=[np.number, "bool"]).columns.tolist()
unhandled_features = set(model_input.columns) - set(categorical_features) - set(numeric_features)
if unhandled_features:
    raise TypeError(f"Add preprocessing for unsupported feature types: {sorted(unhandled_features)}")
model_suite = build_model_suite(numeric_features, categorical_features)
print("Familias de modelos listas:", [ETIQUETAS_VALORES.get(nombre, nombre) for nombre in model_suite])
# En la sensibilidad por deduplicación se usan folds deduplicados para entrenar y los mismos folds principales para validar. No se usa el holdout.

Compara modelos en las mismas particiones temporales, también con el conjunto deduplicado.

In [ ]:
deduplicated_temporal_folds, deduplicated_fold_summary, _ = make_purged_rolling_folds(
    train_df_dedup_sensitivity,
    validation_windows=temporal_validation_windows,
)

primary_cv_results, primary_capacity_results = evaluate_temporal_model_suite(
    train_df, temporal_folds, model_suite, "primary")
deduplicated_train_cv_results, deduplicated_train_capacity_results = evaluate_temporal_model_suite(
    train_df_dedup_sensitivity,
    deduplicated_temporal_folds,
    model_suite,
    "deduplicated_train",
    validation_data=train_df,
    validation_folds=temporal_folds,
)

cv_results = pd.concat([primary_cv_results, deduplicated_train_cv_results], ignore_index=True)
cv_summary = cv_results.groupby(["analysis", "model"], as_index=False).agg(
    average_precision_mean=("average_precision", "mean"),
    average_precision_std=("average_precision", "std"),
    random_baseline_average_precision=("random_baseline_average_precision", "mean"),
)
mostrar_en_espanol(cv_summary.sort_values(["analysis", "average_precision_mean"], ascending=[True, False]))

capacity_results = pd.concat(
    [primary_capacity_results, deduplicated_train_capacity_results], ignore_index=True)
capacity_summary = capacity_results.groupby(
    ["analysis", "model", "fraction_alerted"], as_index=False).agg(
        precision_at_k=("precision_at_k", "mean"),
        recall_at_k=("recall_at_k", "mean"),
        alerts_per_1000=("alerts_per_1000", "mean"),
        random_precision_reference=("random_precision_reference", "mean"),
        random_recall_reference=("random_recall_reference", "mean"),
    )
mostrar_en_espanol(capacity_summary)

### Búsqueda anidada
Ajusta los candidatos dentro de particiones temporales de entrenamiento.

In [ ]:
TUNING_CANDIDATES = {
    "Random forest": [
        {"classifier__n_estimators": 100, "classifier__max_features": "sqrt", "classifier__min_samples_leaf": 2},
        {"classifier__n_estimators": 100, "classifier__max_features": "sqrt", "classifier__min_samples_leaf": 5},
        {"classifier__n_estimators": 100, "classifier__max_features": 0.5, "classifier__min_samples_leaf": 2},
    ],
    "Gradient boosting": [
        {"classifier__n_estimators": 100, "classifier__learning_rate": 0.05, "classifier__max_depth": 2},
        {"classifier__n_estimators": 150, "classifier__learning_rate": 0.05, "classifier__max_depth": 2},
        {"classifier__n_estimators": 100, "classifier__learning_rate": 0.1, "classifier__max_depth": 2},
    ],
}

def run_nested_temporal_search(primary_data, deduplicated_data, outer_folds,
                               deduplicated_outer_folds, validation_data, models,
                               n_inner_splits=3):
    """Selecciona hiperparámetros en folds internos y estima su rendimiento en folds externos."""
    from sklearn.base import clone

    if len(outer_folds) != len(deduplicated_outer_folds):
        raise ValueError("Primary and deduplicated data must have matching outer folds")
    candidate_rows = []
    outer_rows = []
    capacity_frames = []

    for outer_number, (primary_fold, deduplicated_fold) in enumerate(
            zip(outer_folds, deduplicated_outer_folds), start=1):
        primary_train_indices, validation_indices = primary_fold
        deduplicated_train_indices, _ = deduplicated_fold
        primary_outer_train = primary_data.iloc[primary_train_indices].copy()
        deduplicated_outer_train = deduplicated_data.iloc[deduplicated_train_indices].copy()
        outer_validation = validation_data.iloc[validation_indices].copy()
        inner_folds, _, _ = make_purged_rolling_folds(
            primary_outer_train,
            n_splits=n_inner_splits,
            initial_train_fraction=0.5,
        )

        for model_name, candidates in TUNING_CANDIDATES.items():
            model_scores = []
            for candidate_number, parameters in enumerate(candidates, start=1):
                candidate = clone(models[model_name])
                candidate.set_params(**parameters)
                inner_results, _ = evaluate_temporal_model_suite(
                    primary_outer_train,
                    inner_folds,
                    {model_name: candidate},
                    f"outer_{outer_number}_inner",
                )
                mean_inner_ap = inner_results["average_precision"].mean()
                model_scores.append((mean_inner_ap, candidate_number, parameters))
                candidate_rows.append({
                    "outer_fold": outer_number,
                    "model": model_name,
                    "candidate": candidate_number,
                    "parameters": parameters,
                    "inner_average_precision_mean": mean_inner_ap,
                    "inner_average_precision_std": inner_results["average_precision"].std(),
                })

            best_inner_ap, best_candidate_number, best_parameters = max(model_scores, key=lambda row: row[0])
            candidate_rows[-len(candidates) + best_candidate_number - 1]["selected"] = True
            chosen_model = clone(models[model_name])
            chosen_model.set_params(**best_parameters)

            primary_outer_fold = [(np.arange(len(primary_outer_train)), np.arange(len(outer_validation)))]
            primary_result, primary_capacity = evaluate_temporal_model_suite(
                primary_outer_train,
                primary_outer_fold,
                {model_name: chosen_model},
                "nested_primary",
                validation_data=outer_validation,
                validation_folds=primary_outer_fold,
            )
            deduplicated_outer_fold = [(np.arange(len(deduplicated_outer_train)), np.arange(len(outer_validation)))]
            deduplicated_result, deduplicated_capacity = evaluate_temporal_model_suite(
                deduplicated_outer_train,
                deduplicated_outer_fold,
                {model_name: chosen_model},
                "nested_deduplicated_train",
                validation_data=outer_validation,
                validation_folds=primary_outer_fold,
            )

            for analysis, result, capacity in [
                ("nested_primary", primary_result, primary_capacity),
                ("nested_deduplicated_train", deduplicated_result, deduplicated_capacity),
            ]:
                result_row = result.iloc[0].to_dict()
                result_row.update({
                    "analysis": analysis,
                    "outer_fold": outer_number,
                    "selected_candidate": best_candidate_number,
                    "selected_parameters": best_parameters,
                    "best_inner_average_precision": best_inner_ap,
                })
                outer_rows.append(result_row)
                capacity["outer_fold"] = outer_number
                capacity["selected_parameters"] = str(best_parameters)
                capacity_frames.append(capacity)

    return (pd.DataFrame(candidate_rows), pd.DataFrame(outer_rows),
            pd.concat(capacity_frames, ignore_index=True))

nested_candidate_results, nested_outer_results, nested_capacity_results = run_nested_temporal_search(
    train_df,
    train_df_dedup_sensitivity,
    temporal_folds,
    deduplicated_temporal_folds,
    train_df,
    model_suite,
)

nested_outer_summary = nested_outer_results.groupby(["analysis", "model"], as_index=False).agg(
    average_precision_mean=("average_precision", "mean"),
    average_precision_std=("average_precision", "std"),
)
mostrar_en_espanol(nested_outer_summary)
mostrar_en_espanol(nested_candidate_results[nested_candidate_results["selected"] == True])
mostrar_en_espanol(nested_capacity_results.groupby(
    ["analysis", "model", "fraction_alerted"], as_index=False).agg(
        precision_at_k=("precision_at_k", "mean"),
        recall_at_k=("recall_at_k", "mean"),
        alerts_per_1000=("alerts_per_1000", "mean"),
    ))

### Selección del modelo
Selecciona el candidato con resultados de validación temporal.

In [ ]:
from sklearn.base import clone

# La selección de hiperparámetros ocurre solo en train; después se ajustan los candidatos sobre todo train.
final_training_rows = []
final_fitted_models = {}

for model_name in ("Gradient boosting", "Random forest"):
    candidate_scores = []
    for candidate_number, parameters in enumerate(TUNING_CANDIDATES[model_name], start=1):
        candidate = clone(model_suite[model_name])
        candidate.set_params(**parameters)
        fold_results, _ = evaluate_temporal_model_suite(
            train_df,
            temporal_folds,
            {model_name: candidate},
            f"final_train_selection_{model_name}",
        )
        mean_ap = float(fold_results["average_precision"].mean())
        std_ap = float(fold_results["average_precision"].std())
        candidate_scores.append((mean_ap, candidate_number, parameters))
        final_training_rows.append({
            "model": model_name,
            "candidate": candidate_number,
            "parameters": parameters,
            "average_precision_mean": mean_ap,
            "average_precision_std": std_ap,
        })

    best_mean_ap, best_candidate_number, best_parameters = max(
        candidate_scores, key=lambda row: row[0]
    )
    best_row = next(
        row for row in final_training_rows
        if row["model"] == model_name and row["candidate"] == best_candidate_number
    )
    best_row["selected"] = True

    final_model = clone(model_suite[model_name])
    final_model.set_params(**best_parameters)
    final_model.fit(train_df[MODEL_FEATURE_COLS], train_df[TARGET].to_numpy(dtype=int))
    final_fitted_models[model_name] = final_model

    print(
        f"{model_name}: candidato {best_candidate_number}; "
        f"AP media de selección={best_mean_ap:.4f}; parámetros={best_parameters}"
    )

final_training_selection = pd.DataFrame(final_training_rows)
mostrar_en_espanol(final_training_selection.sort_values(
    ["model", "average_precision_mean"], ascending=[True, False]
))

# Calibración del modelo ganador con validación temporal purgada dentro de train.
from sklearn.calibration import CalibratedClassifierCV

primary_model_name = "Gradient boosting"
primary_selection = final_training_selection.loc[
    final_training_selection["model"] == primary_model_name
].query("selected == True").iloc[0]
primary_model = clone(model_suite[primary_model_name])
primary_model.set_params(**primary_selection["parameters"])
# La calibración sigmoid se ajusta con folds temporales internos, sin consultar el holdout.
primary_model_calibrated = CalibratedClassifierCV(
    estimator=primary_model,
    method="sigmoid",
    cv=temporal_folds,
)
primary_model_calibrated.fit(
    train_df[MODEL_FEATURE_COLS],
    train_df[TARGET].to_numpy(dtype=int),
)
print("Calibración sigmoide ajustada con particiones temporales del entrenamiento. El conjunto de prueba permanece cerrado.")
final_model = primary_model_calibrated
final_model_name = primary_model_name

### Calibración
Ajusta y evalúa probabilidades usando solo datos de entrenamiento.

### Ablación y selección de variables
Compara variantes del modelo en particiones temporales de entrenamiento.

In [ ]:
def evaluate_feature_set(data, folds, estimator, feature_columns, analysis_name):
    """Compara un conjunto de variables con predicciones de validación fuera de fold."""
    from sklearn.base import clone
    from sklearn.metrics import average_precision_score

    feature_columns = list(feature_columns)
    if not feature_columns:
        raise ValueError("At least one feature is required")

    X = data[feature_columns]
    y = data[TARGET].to_numpy(dtype=int)
    rows = []

    for fold_number, (train_indices, validation_indices) in enumerate(folds, start=1):
        fitted_model = clone(estimator)
        X_train = X.iloc[train_indices]
        X_validation = X.iloc[validation_indices]
        y_train = y[train_indices]
        y_validation = y[validation_indices]
        fitted_model.fit(X_train, y_train)
        if hasattr(fitted_model, "predict_proba"):
            scores = fitted_model.predict_proba(X_validation)[:, 1]
        else:
            scores = fitted_model.decision_function(X_validation)

        row = {
            "analysis": analysis_name,
            "fold": fold_number,
            "average_precision": average_precision_score(y_validation, scores),
            "n_features": len(feature_columns),
        }
        if "select" in getattr(fitted_model, "named_steps", {}):
            row["selected_transformed_features"] = int(
                fitted_model.named_steps["select"].get_support().sum()
            )
        rows.append(row)

    return pd.DataFrame(rows)


# En la ablation se retira cada familia de variables y se evalúa en los mismos folds temporales.
# Cada análisis elimina una familia derivada para medir su aporte incremental en los mismos folds.
ENGINEERED_FEATURE_GROUPS = {
    "composicion_y_estadia": ["total_nights", "total_guests", "has_children"],
    "calendario": [
        "arrival_month", "arrival_dow", "arrival_is_weekend",
        "arrival_month_sin", "arrival_month_cos",
    ],
    "anticipacion": ["lead_time_band"],
    "historial": ["prev_cancel_ratio"],
}

ablation_feature_sets = {"completo": MODEL_FEATURE_COLS.copy()}
for group_name, group_columns in ENGINEERED_FEATURE_GROUPS.items():
    ablation_feature_sets[f"sin_{group_name}"] = [
        column for column in MODEL_FEATURE_COLS if column not in group_columns
    ]

ablation_rows = []
for analysis_name, feature_columns in ablation_feature_sets.items():
    feature_frame = train_df[feature_columns]
    numeric_columns = feature_frame.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_columns = feature_frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    subset_suite = build_model_suite(numeric_columns, categorical_columns)
    ablation_model = clone(subset_suite["Gradient boosting"])
    ablation_model.set_params(**primary_selection["parameters"])
    ablation_rows.append(
        evaluate_feature_set(
            train_df, temporal_folds, ablation_model, feature_columns, analysis_name
        )
    )

ablation_results = pd.concat(ablation_rows, ignore_index=True)
ablation_summary = ablation_results.groupby("analysis", as_index=False).agg(
    average_precision_mean=("average_precision", "mean"),
    average_precision_std=("average_precision", "std"),
)
ablation_summary["delta_vs_full"] = (
    ablation_summary["average_precision_mean"]
    - ablation_summary.loc[ablation_summary["analysis"] == "completo", "average_precision_mean"].iloc[0]
)
mostrar_en_espanol(ablation_summary.sort_values("average_precision_mean", ascending=False))


# SelectFromModel se ajusta dentro de cada fold de entrenamiento, después del preprocesamiento.
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.base import clone

rf_preprocessor = clone(model_suite["Random forest"].named_steps["preprocess"])
feature_selection_model = Pipeline([
    ("preprocess", rf_preprocessor),
    ("select", SelectFromModel(
        ExtraTreesClassifier(
            n_estimators=100,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED,
        ),
        threshold="median",
    )),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED,
    )),
])

selection_results = evaluate_feature_set(
    train_df,
    temporal_folds,
    feature_selection_model,
    MODEL_FEATURE_COLS,
    "random_forest_select_from_model",
)
mostrar_en_espanol(selection_results)
print(
    "La ablation y la selección son análisis internos sobre train; "
    "no constituyen una evaluación independiente ni usan el test reservado."
)

### Demostración de fuga
Mide el efecto de incluir variables posteriores al resultado; la variante con fuga no es válida.

In [ ]:
# La variante con fuga se compara solo en train para demostrar el efecto de incluir datos posteriores al resultado.
leaky_feature_columns = MODEL_FEATURE_COLS + LEAK_COLS
leaky_input = train_df[leaky_feature_columns]
leaky_categorical_features = leaky_input.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
leaky_numeric_features = leaky_input.select_dtypes(
    include=[np.number, "bool"]
).columns.tolist()
leaky_model_suite = build_model_suite(
    leaky_numeric_features, leaky_categorical_features
)

# Se mantienen algoritmo, parámetros y folds; solo cambian las columnas que contienen fuga.
honest_model_for_leak_test = clone(model_suite["Gradient boosting"])
honest_model_for_leak_test.set_params(**primary_selection["parameters"])
leaky_model_for_test = clone(leaky_model_suite["Gradient boosting"])
leaky_model_for_test.set_params(**primary_selection["parameters"])

honest_leak_results = evaluate_feature_set(
    train_df,
    temporal_folds,
    honest_model_for_leak_test,
    MODEL_FEATURE_COLS,
    "honesto_sin_fuga",
)
leaky_results = evaluate_feature_set(
    train_df,
    temporal_folds,
    leaky_model_for_test,
    leaky_feature_columns,
    "deliberadamente_con_fuga",
)

leak_results = pd.concat([honest_leak_results, leaky_results], ignore_index=True)
leak_summary = leak_results.groupby("analysis", as_index=False).agg(
    average_precision_mean=("average_precision", "mean"),
    average_precision_std=("average_precision", "std"),
)
honest_ap = leak_summary.loc[
    leak_summary["analysis"] == "honesto_sin_fuga", "average_precision_mean"
].iloc[0]
leaky_ap = leak_summary.loc[
    leak_summary["analysis"] == "deliberadamente_con_fuga", "average_precision_mean"
].iloc[0]
leak_summary["delta_vs_honest"] = leak_summary["average_precision_mean"] - honest_ap
print(f"Cambio de AP por la fuga deliberada: {leaky_ap - honest_ap:+.4f}")
mostrar_en_espanol(leak_summary)
mostrar_en_espanol(leak_results)
print(
    "La variante con fuga es solo una demostración metodológica; "
    "no es un modelo válido ni debe evaluarse sobre el test."
)

### Costos y umbral
Compara escenarios relativos; no son costos reales ni una recomendación operativa.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss, log_loss

# Las probabilidades fuera de fold permiten evaluar calibración sin ajustar y evaluar sobre las mismas filas.
# Cada fold externo se calibra dentro de su train y predice únicamente su ventana futura.
calibration_oof_rows = []
for outer_fold_number, (outer_train_indices, outer_validation_indices) in enumerate(
    temporal_folds, start=1
):
    outer_train = train_df.iloc[outer_train_indices].reset_index(drop=True)
    outer_validation = train_df.iloc[outer_validation_indices]
    inner_folds, _, _ = make_purged_rolling_folds(
        outer_train,
        n_splits=3,
        initial_train_fraction=0.5,
    )
    selected_candidate = nested_candidate_results.loc[
        (nested_candidate_results["outer_fold"] == outer_fold_number)
        & (nested_candidate_results["model"] == "Gradient boosting")
        & (nested_candidate_results["selected"] == True)
    ].iloc[0]
    fold_estimator = clone(model_suite["Gradient boosting"])
    fold_estimator.set_params(**selected_candidate["parameters"])
    fold_calibrated = CalibratedClassifierCV(
        estimator=fold_estimator,
        method="sigmoid",
        cv=inner_folds,
    )
    fold_calibrated.fit(
        outer_train[MODEL_FEATURE_COLS],
        outer_train[TARGET].to_numpy(dtype=int),
    )
    fold_probabilities = fold_calibrated.predict_proba(
        outer_validation[MODEL_FEATURE_COLS]
    )[:, 1]
    calibration_oof_rows.append(pd.DataFrame({
        "outer_fold": outer_fold_number,
        "y_true": outer_validation[TARGET].to_numpy(dtype=int),
        "y_probability": fold_probabilities,
    }))

calibration_oof = pd.concat(calibration_oof_rows, ignore_index=True)
calibration_metrics = pd.DataFrame([{
    "brier_score": brier_score_loss(
        calibration_oof["y_true"], calibration_oof["y_probability"]
    ),
    "log_loss": log_loss(
        calibration_oof["y_true"], calibration_oof["y_probability"], labels=[0, 1]
    ),
    "validation_rows": len(calibration_oof),
}])
mostrar_en_espanol(calibration_metrics)

fraction_of_positives, mean_predicted_value = calibration_curve(
    calibration_oof["y_true"],
    calibration_oof["y_probability"],
    n_bins=10,
    strategy="quantile",
)
plt.plot(mean_predicted_value, fraction_of_positives, marker="o", label="Gradient Boosting calibrado")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibración perfecta")
plt.xlabel("Probabilidad media predicha")
plt.ylabel("Fracción observada de cancelaciones")
plt.title("Calibración temporal fuera de muestra (train)")
plt.legend()
plt.tight_layout()
plt.show()

cost_scenarios = [
    {"scenario": "FP=1, FN=1", "cost_fp": 1.0, "cost_fn": 1.0},
    {"scenario": "FP=1, FN=2", "cost_fp": 1.0, "cost_fn": 2.0},
    {"scenario": "FP=1, FN=5", "cost_fp": 1.0, "cost_fn": 5.0},
    {"scenario": "FP=1, FN=10", "cost_fp": 1.0, "cost_fn": 10.0},
    {"scenario": "FP=2, FN=1", "cost_fp": 2.0, "cost_fn": 1.0},
]
y_cost = calibration_oof["y_true"].to_numpy(dtype=int)
p_cost = calibration_oof["y_probability"].to_numpy(dtype=float)
cost_rows = []
for scenario in cost_scenarios:
    threshold = scenario["cost_fp"] / (scenario["cost_fp"] + scenario["cost_fn"])
    predicted_cancel = p_cost >= threshold
    false_positives = int(((predicted_cancel == 1) & (y_cost == 0)).sum())
    false_negatives = int(((predicted_cancel == 0) & (y_cost == 1)).sum())
    cost_rows.append({
        **scenario,
        "threshold": threshold,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "normalized_cost_per_reservation": (
            scenario["cost_fp"] * false_positives
            + scenario["cost_fn"] * false_negatives
        ) / len(y_cost),
        "alerts_per_1000": predicted_cancel.mean() * 1000,
    })
cost_sensitivity = pd.DataFrame(cost_rows)
mostrar_en_espanol(cost_sensitivity)
print(
    "Los costos son unidades relativas hipotéticas. No elegir un escenario ni un umbral "
    "operativo hasta acordarlo con la cátedra o contar con costos del negocio."
)

## 7. Evaluación final
El conjunto de prueba ya se evaluó una sola vez y no es completamente ciego. La evaluación permanece desactivada; no lo abras ni lo uses para ajustar el modelo.

In [ ]:
# El conjunto de prueba ya se evaluó una sola vez; permanece cerrado para evitar una segunda evaluación.
RUN_FINAL_TEST_EVALUATION = False
# Umbral del escenario académico acordado: costos relativos FP=1 y FN=1.
FINAL_THRESHOLD = 0.5

if not RUN_FINAL_TEST_EVALUATION:
    print(
        "La evaluación final ya se realizó; no se debe volver a abrir el holdout."
    )
else:
    if FINAL_THRESHOLD is None or not 0 < float(FINAL_THRESHOLD) < 1:
        raise ValueError("Set FINAL_THRESHOLD to the agreed value in (0, 1) before opening test")

    from sklearn.metrics import (
        average_precision_score,
        brier_score_loss,
        classification_report,
        confusion_matrix,
        f1_score,
        log_loss,
        precision_recall_curve,
        precision_score,
        recall_score,
        roc_auc_score,
    )

    # Esta es la única celda que lee el holdout reservado.
    final_test_df = pd.read_csv("test_holdout.csv", parse_dates=["arrival_date"])
    final_test_df = add_stateless_features(final_test_df)
    X_test_final = final_test_df[MODEL_FEATURE_COLS]
    y_test_final = final_test_df[TARGET].to_numpy(dtype=int)
    test_probabilities = final_model.predict_proba(X_test_final)[:, 1]
    test_predictions = (test_probabilities >= float(FINAL_THRESHOLD)).astype(int)

    final_test_metrics = pd.DataFrame([{
        "average_precision": average_precision_score(y_test_final, test_probabilities),
        "roc_auc": roc_auc_score(y_test_final, test_probabilities),
        "precision": precision_score(y_test_final, test_predictions, zero_division=0),
        "recall": recall_score(y_test_final, test_predictions, zero_division=0),
        "f1": f1_score(y_test_final, test_predictions, zero_division=0),
        "brier_score": brier_score_loss(y_test_final, test_probabilities),
        "log_loss": log_loss(y_test_final, test_probabilities, labels=[0, 1]),
        "threshold": float(FINAL_THRESHOLD),
        "test_rows": len(y_test_final),
    }])
    mostrar_en_espanol(final_test_metrics)
    print("Matriz de confusión (filas = real; columnas = predicho):")
    mostrar_en_espanol(pd.DataFrame(
        confusion_matrix(y_test_final, test_predictions),
        index=["Real: no cancela", "Real: cancela"],
        columns=["Predicho: no cancela", "Predicho: cancela"],
    ))
    reporte = pd.DataFrame(classification_report(
        y_test_final,
        test_predictions,
        target_names=["No cancelada", "Cancelada"],
        output_dict=True,
        zero_division=0,
    )).transpose()
    reporte = reporte.rename(columns={"f1-score": "F1", "support": "reservas"})
    mostrar_en_espanol(reporte)

    test_average_precision, test_capacity_scenarios = capacity_scenario_metrics(
        y_test_final, test_probabilities
    )
    print(f"Average Precision en la prueba: {test_average_precision:.4f}")
    mostrar_en_espanol(test_capacity_scenarios)

    precision_points, recall_points, _ = precision_recall_curve(
        y_test_final, test_probabilities
    )
    plt.plot(recall_points, precision_points)
    plt.xlabel("Exhaustividad")
    plt.ylabel("Precisión")
    plt.title("Curva de precisión y exhaustividad en la prueba reservada")
    plt.tight_layout()
    plt.show()

    bootstrap_rng = np.random.default_rng(SEED)
    bootstrap_rows = []
    bootstrap_replicates = 1000
    for _ in range(bootstrap_replicates):
        sample_indices = bootstrap_rng.integers(0, len(y_test_final), len(y_test_final))
        y_sample = y_test_final[sample_indices]
        probability_sample = test_probabilities[sample_indices]
        prediction_sample = test_predictions[sample_indices]
        if np.unique(y_sample).size < 2:
            continue
        bootstrap_rows.append({
            "average_precision": average_precision_score(y_sample, probability_sample),
            "roc_auc": roc_auc_score(y_sample, probability_sample),
            "precision": precision_score(y_sample, prediction_sample, zero_division=0),
            "recall": recall_score(y_sample, prediction_sample, zero_division=0),
            "f1": f1_score(y_sample, prediction_sample, zero_division=0),
        })

    bootstrap_frame = pd.DataFrame(bootstrap_rows)
    confidence_intervals = pd.DataFrame([
        {
            "metric": metric,
            "estimate": final_test_metrics.loc[0, metric],
            "ci_2_5%": bootstrap_frame[metric].quantile(0.025),
            "ci_97_5%": bootstrap_frame[metric].quantile(0.975),
            "valid_replicates": len(bootstrap_frame),
        }
        for metric in ["average_precision", "roc_auc", "precision", "recall", "f1"]
    ])
    mostrar_en_espanol(confidence_intervals)
    print(
        "Interpretar estos resultados con la limitación de que el holdout no es "
        "completamente ciego; no reutilizarlo para seleccionar modelos o umbrales."
    )

### Interpretabilidad y subgrupos
Resume errores e importancia de variables con validación de entrenamiento.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

# Las ventanas temporales de validación producen predicciones fuera de fold para train.
# Los análisis de errores e interpretabilidad usan filas OOF y no el holdout final.
oof_validation_indices = np.concatenate([
    validation_indices for _, validation_indices in temporal_folds
])
if len(oof_validation_indices) != len(calibration_oof):
    raise AssertionError("OOF rows must align with the temporal validation windows")
oof_error_review = train_df.iloc[oof_validation_indices].copy().reset_index(drop=True)
oof_error_review["y_true"] = calibration_oof["y_true"].to_numpy(dtype=int)
oof_error_review["cancellation_probability"] = calibration_oof["y_probability"].to_numpy(dtype=float)
review_columns = [
    column for column in [
        "hotel", "lead_time", "arrival_month", "market_segment",
        "total_nights", "total_guests", "y_true", "cancellation_probability",
    ]
    if column in oof_error_review.columns
]
missed_cancellations = oof_error_review.loc[
    oof_error_review["y_true"] == 1
].nsmallest(15, "cancellation_probability")
false_alarm_risks = oof_error_review.loc[
    oof_error_review["y_true"] == 0
].nlargest(15, "cancellation_probability")
print("Cancelaciones reales con menor score (riesgo de omisión):")
mostrar_en_espanol(missed_cancellations[review_columns])
print("Reservas no canceladas con mayor puntuación (riesgo de falsa alerta):")
mostrar_en_espanol(false_alarm_risks[review_columns])

# Métricas descriptivas por subgrupo; se omiten grupos con menos de 200 reservas.
subgroup_rows = []
row_offset = 0
for fold_number, (_, validation_indices) in enumerate(temporal_folds, start=1):
    fold_size = len(validation_indices)
    fold_scores = calibration_oof.iloc[row_offset:row_offset + fold_size]
    fold_data = train_df.iloc[validation_indices].reset_index(drop=True)
    row_offset += fold_size
    for group_column in ["hotel", "market_segment", "country"]:
        if group_column not in fold_data.columns:
            continue
        for group_value, group_indices in fold_data.groupby(
            group_column, dropna=False, observed=True
        ).groups.items():
            group_indices = np.asarray(list(group_indices), dtype=int)
            if len(group_indices) < 200:
                continue
            y_group = fold_scores.iloc[group_indices]["y_true"].to_numpy(dtype=int)
            score_group = fold_scores.iloc[group_indices]["y_probability"].to_numpy(dtype=float)
            if np.unique(y_group).size < 2:
                continue
            group_ap, group_capacity = capacity_scenario_metrics(y_group, score_group)
            top_ten = group_capacity.loc[
                group_capacity["fraction_alerted"] == 0.10
            ].iloc[0]
            subgroup_rows.append({
                "fold": fold_number,
                "group": group_column,
                "value": "(missing)" if pd.isna(group_value) else str(group_value),
                "rows": len(group_indices),
                "average_precision": group_ap,
                "precision_at_10_percent": top_ten["precision_at_k"],
                "recall_at_10_percent": top_ten["recall_at_k"],
            })
subgroup_metrics = pd.DataFrame(subgroup_rows)
mostrar_en_espanol(subgroup_metrics.sort_values(["group", "fold", "rows"], ascending=[True, True, False]))

# Importancia por permutación en cada fold temporal de validación, no en el test final.
permutation_rows = []
for fold_number, (train_indices, validation_indices) in enumerate(temporal_folds, start=1):
    selected_candidate = nested_candidate_results.loc[
        (nested_candidate_results["outer_fold"] == fold_number)
        & (nested_candidate_results["model"] == "Gradient boosting")
        & (nested_candidate_results["selected"] == True)
    ].iloc[0]
    fold_model = clone(model_suite["Gradient boosting"])
    fold_model.set_params(**selected_candidate["parameters"])
    fold_model.fit(
        train_df.iloc[train_indices][MODEL_FEATURE_COLS],
        train_df.iloc[train_indices][TARGET].to_numpy(dtype=int),
    )
    validation_sample = train_df.iloc[validation_indices]
    if len(validation_sample) > 2500:
        validation_sample = validation_sample.sample(n=2500, random_state=SEED)
    importance = permutation_importance(
        fold_model,
        validation_sample[MODEL_FEATURE_COLS],
        validation_sample[TARGET].to_numpy(dtype=int),
        scoring="average_precision",
        n_repeats=3,
        random_state=SEED,
        n_jobs=1,
    )
    permutation_rows.extend({
        "fold": fold_number,
        "feature": feature,
        "importance": float(mean_importance),
    } for feature, mean_importance in zip(
        MODEL_FEATURE_COLS, importance.importances_mean
    ))

permutation_importance_by_fold = pd.DataFrame(permutation_rows)
permutation_importance_summary = permutation_importance_by_fold.groupby(
    "feature", as_index=False
).agg(
    importance_mean=("importance", "mean"),
    importance_std_between_folds=("importance", "std"),
)
permutation_importance_summary = permutation_importance_summary.sort_values(
    "importance_mean", ascending=False
)
mostrar_en_espanol(permutation_importance_summary.head(20))
permutation_importance_summary.head(15).sort_values("importance_mean").plot.barh(
    x="feature", y="importance_mean", xerr="importance_std_between_folds",
    legend=False, figsize=(8, 6),
)
plt.xlabel("Caída de Average Precisión al permutar")
plt.title("Importancia por permutación en folds temporales de train")
plt.tight_layout()
plt.show()

# La dependencia parcial usa una muestra de train y no consulta el holdout reservado.
pdp_sample = train_df[MODEL_FEATURE_COLS].sample(
    n=min(2000, len(train_df)), random_state=SEED
)
pdp_features = ["lead_time", "total_nights"]
pdp_sample[pdp_features] = pdp_sample[pdp_features].astype(float)
PartialDependenceDisplay.from_estimator(
    final_model,
    pdp_sample,
    features=pdp_features,
    response_method="predict_proba",
    target=1,
)
plt.tight_layout()
plt.show()
print(
    "Los resultados por subgrupo son descriptivos; no implican causalidad ni "
    "garantizan desempeño estable en otros hoteles o períodos."
)

## 8. Riesgos y uso responsable
No desplegar ni usar para sobreventa sin validación operativa y supervisión humana.

In [ ]:
if FINAL_THRESHOLD is None:
    print(
        "Artefactos pendientes: define FINAL_THRESHOLD tras acordar el escenario "
        "de costos con el equipo/cátedra. No se asigna un umbral arbitrario."
    )
else:
    import joblib
    from pathlib import Path

    MODEL_ARTIFACT_PATH = Path("hotel_cancellation_pipeline.joblib")
    joblib.dump(final_model, MODEL_ARTIFACT_PATH)

    predict_script_template = '''import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

MODEL_PATH = Path(__file__).with_name("hotel_cancellation_pipeline.joblib")
MODEL_FEATURES = __MODEL_FEATURES__
CANCELLATION_THRESHOLD = __FINAL_THRESHOLD__
MONTHS = {name: index for index, name in enumerate(
    ["January", "February", "March", "April", "May", "June",
     "July", "August", "September", "October", "November", "December"],
    start=1,
)}


def prepare_features(raw_data):
    data = raw_data.copy()
    if "arrival_date" not in data.columns:
        data["arrival_date"] = pd.to_datetime(dict(
            year=data["arrival_date_year"],
            month=data["arrival_date_month"].map(MONTHS),
            day=data["arrival_date_day_of_month"],
        ))
    else:
        data["arrival_date"] = pd.to_datetime(data["arrival_date"])

    data["total_nights"] = data["stays_in_weekend_nights"] + data["stays_in_week_nights"]
    data["total_guests"] = data["adults"] + data["children"].fillna(0) + data["babies"]
    data["has_children"] = (
        (data["children"].fillna(0) + data["babies"]) > 0
    ).astype(int)
    data["arrival_month"] = data["arrival_date"].dt.month
    data["arrival_dow"] = data["arrival_date"].dt.day_of_week
    data["arrival_is_weekend"] = data["arrival_dow"].isin([5, 6]).astype(int)
    data["arrival_month_sin"] = np.sin(2 * np.pi * data["arrival_month"] / 12)
    data["arrival_month_cos"] = np.cos(2 * np.pi * data["arrival_month"] / 12)
    data["lead_time_band"] = pd.cut(
        data["lead_time"],
        bins=[-1, 7, 30, 90, 180, 365, np.inf],
        labels=["0-7", "8-30", "31-90", "91-180", "181-365", "366+"],
    ).astype("object")
    prior_bookings = data["previous_cancellations"] + data["previous_bookings_not_canceled"]
    data["prev_cancel_ratio"] = data["previous_cancellations"].div(
        prior_bookings.where(prior_bookings > 0)
    ).fillna(0)
    return data


def predict_reservations(raw_data, model_path=MODEL_PATH):
    model = joblib.load(model_path)
    features = prepare_features(raw_data)
    probabilities = model.predict_proba(features[MODEL_FEATURES])[:, 1]
    result = raw_data.copy()
    result["cancellation_probability"] = probabilities
    result["predicted_cancellation"] = (
        probabilities >= CANCELLATION_THRESHOLD
    ).astype(int)
    return result


if __name__ == "__main__":
    if len(sys.argv) != 2:
        raise SystemExit("Usage: python predict.py reservations.csv")
    predictions = predict_reservations(pd.read_csv(sys.argv[1]))
    mostrar_en_espanol(predictions)
'''
    predict_script = predict_script_template.replace(
        "__MODEL_FEATURES__", repr(MODEL_FEATURE_COLS)
    ).replace("__FINAL_THRESHOLD__", repr(float(FINAL_THRESHOLD)))
    Path("predict.py").write_text(predict_script, encoding="utf-8")

    example_reservation = train_df.iloc[[0]].drop(columns=[TARGET])
    example_features = add_stateless_features(example_reservation)
    example_probability = final_model.predict_proba(
        example_features[MODEL_FEATURE_COLS]
    )[:, 1]
    mostrar_en_espanol(pd.DataFrame({
        "cancellation_probability": example_probability,
        "predicted_cancellation": (example_probability >= FINAL_THRESHOLD).astype(int),
    }))
    print(f"Pipeline guardado en: {MODEL_ARTIFACT_PATH}")
    print("Script de ejemplo generado en: predict.py")

## 9. Uso de inteligencia artificial
La declaración completa está en el informe Word.

In [ ]:
print("La declaración completa de uso de inteligencia artificial está en el informe Word.")

## Anexo Word opcional
Estas celdas generan una copia del informe si el DOCX está disponible. No son necesarias para el análisis.

In [ ]:
from pathlib import Path
from docx import Document

source_docx = Path("Obligatorio Inteligencia Artificial.docx")
updated_docx = Path("Obligatorio Inteligencia Artificial - actualizado.docx")
if not source_docx.exists():
    print(f"Se omite el informe opcional; no se encontró el archivo fuente: {source_docx}")
else:
    report = Document(source_docx)
    report.add_page_break()
    report.add_heading("Actualización de avances", level=1)
    report.add_paragraph(
        "Fecha: 3 de noviembre de 2026. Este anexo conserva el informe original y registra "
        "los cambios metodológicos incorporados al notebook. Los resultados se incluyen "
        "solo cuando las celdas correspondientes se ejecutaron; no se fabrican métricas."
    )
    report.add_heading("Cambios incorporados", level=2)
    for item in [
        "Ablation de familias de variables y selección SelectFromModel dentro de validación temporal.",
        "Experimento controlado con reservation_status y reservation_status_date, comparado con el modelo honesto en los mismos folds de train.",
        "Calibración temporal fuera de muestra y escenarios hipotéticos de costos relativos; no hay costos reales ni umbral operativo acordado.",
        "Evaluacion final del holdout ejecutada una sola vez con umbral 0.50, acordado bajo FP=FN=1; se incluyen matriz de confusion y bootstrap.",
        "Análisis de errores, subgrupos, importancia por permutación y dependencia parcial sobre validaciones temporales de train.",
        "Código para guardar el pipeline, generar predict.py y registrar las versiones instaladas en requirements.txt.",
    ]:
        report.add_paragraph("- " + item)


    def append_results_table(heading, frame, columns=None, max_rows=12):
        report.add_heading(heading, level=2)
        if frame is None or frame.empty:
            report.add_paragraph(
                "Pendiente de ejecución en el notebook; no hay resultados reproducibles disponibles todavía."
            )
            return
        visible = frame if columns is None else frame[[column for column in columns if column in frame.columns]]
        visible = visible.head(max_rows)
        table = report.add_table(rows=1, cols=len(visible.columns))
        for cell, column in zip(table.rows[0].cells, visible.columns):
            cell.text = str(column)
        for _, row in visible.iterrows():
            cells = table.add_row().cells
            for cell, value in zip(cells, row):
                if isinstance(value, (float, np.floating)):
                    cell.text = f"{value:.4f}"
                else:
                    cell.text = str(value)


    append_results_table(
        "Ablation por familias de variables",
        globals().get("ablation_summary"),
    )
    append_results_table(
        "Experimento de fuga deliberada",
        globals().get("leak_summary"),
    )
    append_results_table(
        "Calibración temporal",
        globals().get("calibration_metrics"),
    )
    append_results_table(
        "Sensibilidad a costos relativos (escenarios hipotéticos)",
        globals().get("cost_sensitivity"),
        columns=[
            "scenario", "cost_fp", "cost_fn", "threshold",
            "normalized_cost_per_reservation", "alerts_per_1000",
        ],
    )
    append_results_table(
        "Evaluación final del test y sus intervalos",
        globals().get("confidence_intervals"),
    )

    report.add_heading("Limitaciones y decisiones pendientes", level=2)
    for item in [
        "La fecha estimada de reserva es una aproximación y el test puede no ser completamente ciego por una inspección previa de etiquetas y filas compartidas. No seleccionar modelos ni umbrales con ese conjunto.",
        "El escenario FP=FN=1 y el umbral 0.50 se acordaron para el ejercicio. Son costos relativos hipoteticos, no importes confirmados por un hotel.",
        "Los análisis y la evaluación final se ejecutaron. El holdout no debe volver a abrirse ni usarse para ajustar el modelo.",
        "No recomendar despliegue sin un piloto supervisado, costos operativos, revisión de subgrupos y monitoreo de deriva y calibración.",
    ]:
        report.add_paragraph("- " + item)

    report.add_paragraph(
        "Este anexo registra los resultados reproducibles de la última ejecución. Ante discrepancias con el informe original, usar las cifras de este anexo."
    )
    report.save(updated_docx)
    print(f"Informe actualizado creado sin modificar el original: {updated_docx}")

In [ ]:
if not updated_docx.exists():
    print("Se omite el anexo opcional; no se creó un DOCX actualizado en este entorno.")
else:
    report = Document(updated_docx)
    report.add_heading("Estado de la evaluación", level=2)
    for item in [
        "El notebook se ejecutó con comparación de modelos, calibración, fuga deliberada, ablation, interpretabilidad y evaluación final.",
        "El conjunto de prueba se evaluó una sola vez con el umbral acordado FP=FN=1 (0.50); no volver a abrirlo para seleccionar modelos o umbrales.",
        "Los resultados del holdout se interpretan con cautela porque la partición no es completamente ciega.",
    ]:
        report.add_paragraph("- " + item)
    report.save(updated_docx)
    print("Anexo de aclaraciones guardado en el DOCX actualizado.")